In [1]:
import pandas as pd
import numpy as np
import re

In [3]:
df = pd.read_csv("/content/youtube_comments_zomato.csv")
df.head()

,comment_id,video_id,video_title,channel,author,text,like_count,published_at,reply_count,type
0,UgwueYSKomeDUJT3osx4AaABAg,jUZAh4aHfEc,Zomato IPO - Explained By Assetyogi,Asset Yogi,@AssetYogi,Join All-In-One Video Finance App here: https:...,33,2021-07-10T15:06:46Z,10,top_level
1,UgwueYSKomeDUJT3osx4AaABAg.9PcAiMVsCpG9PcMBgTWp8E,jUZAh4aHfEc,Zomato IPO - Explained By Assetyogi,Asset Yogi,@YogeshKumarenglish,big no for zomato,1,2021-07-10T16:47:01Z,0,reply
2,UgwueYSKomeDUJT3osx4AaABAg.9PcAiMVsCpG9PcSmtDDrmK,jUZAh4aHfEc,Zomato IPO - Explained By Assetyogi,Asset Yogi,@manpreetjassal5321,great,2,2021-07-10T17:44:40Z,0,reply
3,UgwueYSKomeDUJT3osx4AaABAg.9PcAiMVsCpG9PdaQweX6GG,jUZAh4aHfEc,Zomato IPO - Explained By Assetyogi,Asset Yogi,@chandranshpandey1929,most stocks in stock market are valued for sho...,1,2021-07-11T04:19:25Z,0,reply
4,UgwueYSKomeDUJT3osx4AaABAg.9PcAiMVsCpG9Pgq8wd2g2g,jUZAh4aHfEc,Zomato IPO - Explained By Assetyogi,Asset Yogi,@shivamkumarraghav9972,I also waited for you.,0,2021-07-12T10:34:29Z,0,reply


In [4]:
print("=== Null counts (before):")
print(df.isnull().sum())
print()

=== Null counts (before):
comment_id      0
video_id        0
video_title     0
channel         0
author          0
text            0
like_count      0
published_at    0
reply_count     0
type            0
dtype: int64



In [5]:
omment_ids = df["comment_id"].copy()
df.drop(columns=["comment_id", "video_id"], inplace=True)
print("Dropped 'comment_id' and 'video_id' (identifier columns).")

Dropped 'comment_id' and 'video_id' (identifier columns).


In [6]:
df["published_at"] = pd.to_datetime(df["published_at"], utc=True)

In [7]:
df["year"]         = df["published_at"].dt.year
df["month"]        = df["published_at"].dt.month
df["day_of_week"]  = df["published_at"].dt.dayofweek   # 0=Monday … 6=Sunday
df["hour"]         = df["published_at"].dt.hour

In [8]:
df.drop(columns=["published_at"], inplace=True)
print("Parsed 'published_at' → year, month, day_of_week, hour.")

Parsed 'published_at' → year, month, day_of_week, hour.


In [9]:
before = len(df)
df.drop_duplicates(subset=["text"], inplace=True)
print(f"\nDuplicate texts removed: {before - len(df)} (kept first occurrence)")


Duplicate texts removed: 1503 (kept first occurrence)


In [10]:
def clean_text(text: str) -> str:
    """
    NLP-ready cleaning for YouTube comments:
      - Remove URLs
      - Remove @mentions
      - Remove emojis / non-ASCII
      - Remove special characters
      - Collapse whitespace
      - Strip and lowercase
    """
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    # Remove @mentions
    text = re.sub(r"@\w+", "", text)
    # Remove emojis and non-ASCII characters
    text = text.encode("ascii", "ignore").decode("ascii")
    # Remove special characters, keep alphanumerics and basic punctuation
    text = re.sub(r"[^a-zA-Z0-9\s.,!?']", " ", text)
    # Collapse multiple whitespace / newlines
    text = re.sub(r"\s+", " ", text).strip()
    # Lowercase
    text = text.lower()
    return text

df["text_clean"] = df["text"].apply(clean_text)
print("\nSample cleaned text:")
print(df[["text", "text_clean"]].head(3).to_string())


Sample cleaned text:
                                                                                                                                                                                                                                                 text                                                                                                                                                                     text_clean
0  Join All-In-One Video Finance App here: https://vidfin.com/\nEnjoy Free Access to select Financial Insights & 1st Chapter of all the courses.\n\nTo invest in IPO, open your Discount Demat Account here: \n✔ https://bit.ly/demat-trading-account  join all in one video finance app here enjoy free access to select financial insights 1st chapter of all the courses. to invest in ipo, open your discount demat account here
1                                                                                                                                       

In [11]:
df["text_clean"] = df["text_clean"].str.strip()
empty_mask = df["text_clean"].str.len() < 3
print(f"\nComments empty after cleaning: {empty_mask.sum()} — dropping.")
df = df[~empty_mask].copy()


Comments empty after cleaning: 149 — dropping.


In [12]:
df["word_count"] = df["text_clean"].apply(lambda x: len(x.split()))
df["char_count"] = df["text_clean"].apply(len)

print("\nText length stats:")
print(df[["word_count", "char_count"]].describe())


Text length stats:
         word_count    char_count
count  16487.000000  16487.000000
mean      17.404682     94.725602
std       23.879604    133.066217
min        1.000000      3.000000
25%        5.000000     29.000000
50%       11.000000     58.000000
75%       21.000000    116.000000
max     1359.000000   7838.000000


In [13]:
df["is_reply"] = (df["type"] == "reply").astype(int)
df.drop(columns=["type"], inplace=True)
print("\n'type' encoded as binary 'is_reply': 1=reply, 0=top_level")


'type' encoded as binary 'is_reply': 1=reply, 0=top_level


In [14]:
channel_freq = df["channel"].value_counts().to_dict()
df["channel_freq"] = df["channel"].map(channel_freq)
print(f"\nChannel frequency encoding applied (90 unique channels).")
print("Top 5 channels by comment count:")
print(df.groupby("channel")["channel_freq"].first().sort_values(ascending=False).head())


Channel frequency encoding applied (90 unique channels).
Top 5 channels by comment count:
channel
Akshat Shrivastava          7487
CA Rachana Phadke Ranade    2046
Think School                 779
Sahil Bhadviya               638
Fortune India                611
Name: channel_freq, dtype: int64


In [15]:
author_freq = df["author"].value_counts().to_dict()
df["author_comment_count"] = df["author"].map(author_freq)
df.drop(columns=["author"], inplace=True)
print(f"\n'author' replaced with 'author_comment_count' (activity proxy).")


'author' replaced with 'author_comment_count' (activity proxy).


In [16]:
video_freq = df["video_title"].value_counts().to_dict()
df["video_comment_count"] = df["video_title"].map(video_freq)
df.drop(columns=["video_title"], inplace=True)
print("'video_title' replaced with 'video_comment_count' (popularity proxy).")

'video_title' replaced with 'video_comment_count' (popularity proxy).


In [17]:
df["like_count_log"]  = np.log1p(df["like_count"])
df["reply_count_log"] = np.log1p(df["reply_count"])
print("\nLog-transformed 'like_count' and 'reply_count' to handle skew.")


Log-transformed 'like_count' and 'reply_count' to handle skew.


In [18]:
df.reset_index(drop=True, inplace=True)

In [19]:
print("\n=== Preprocessed DataFrame ===")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nNull counts (after):")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)
print("\nSample rows (key columns):")
print(df[["text_clean", "word_count", "is_reply",
          "like_count_log", "channel_freq", "year", "hour"]].head(5).to_string())

# ── 16. Save ─────────────────────────────────────────────────────────────────
df.to_csv("youtube_comments_preprocessed.csv", index=False)
print("\nSaved to youtube_comments_preprocessed.csv")


=== Preprocessed DataFrame ===
Shape: (16487, 17)

Columns: ['channel', 'text', 'like_count', 'reply_count', 'year', 'month', 'day_of_week', 'hour', 'text_clean', 'word_count', 'char_count', 'is_reply', 'channel_freq', 'author_comment_count', 'video_comment_count', 'like_count_log', 'reply_count_log']

Null counts (after):
channel                 0
text                    0
like_count              0
reply_count             0
year                    0
month                   0
day_of_week             0
hour                    0
text_clean              0
word_count              0
char_count              0
is_reply                0
channel_freq            0
author_comment_count    0
video_comment_count     0
like_count_log          0
reply_count_log         0
dtype: int64

Data types:
channel                  object
text                     object
like_count                int64
reply_count               int64
year                      int32
month                     int32
day_of_week   